# Flow Model Notebook Documentation

This notebook builds anomaly-detection models from LANL cyber logs, with a focus on network flow behavior and supporting pipelines for authentication and process logs.

The workflow is:
1. Prepare folders and dataset URLs.
2. Download compressed LANL log files.
3. Load red-team ground-truth events.
4. Sample large logs in chunks and label anomalies.
5. Engineer features per log type.
6. Train XGBoost classifiers and save model artifacts.


## 1) Environment and Dataset Setup


In [1]:
import os
import subprocess

data_dir = './lanl_data'
model_dir = './saved_models'
os.makedirs(data_dir, exist_ok=True)
os.makedirs(model_dir, exist_ok=True)

print("Installing aria2 for multi-threaded downloading")
subprocess.run("sudo apt-get update > /dev/null 2>&1 && sudo apt-get install -y aria2 > /dev/null 2>&1", shell=True)

urls = {
    "redteam.txt.gz": "https://csr.lanl.gov/data-fence/1774313780/_qhbLmkHe22dfLBm4hZ62skbI9A=/cyber1/redteam.txt.gz",
    "auth.txt.gz": "https://csr.lanl.gov/data-fence/1774313780/_qhbLmkHe22dfLBm4hZ62skbI9A=/cyber1/auth.txt.gz",
    "proc.txt.gz": "https://csr.lanl.gov/data-fence/1774313780/_qhbLmkHe22dfLBm4hZ62skbI9A=/cyber1/proc.txt.gz",
    "flows.txt.gz": "https://csr.lanl.gov/data-fence/1774313780/_qhbLmkHe22dfLBm4hZ62skbI9A=/cyber1/flows.txt.gz",
    "dns.txt.gz": "https://csr.lanl.gov/data-fence/1774313780/_qhbLmkHe22dfLBm4hZ62skbI9A=/cyber1/dns.txt.gz"
}

Installing aria2 for multi-threaded downloading


## 2) File Download Execution


In [2]:
for filename, url in urls.items():
    dest_path = os.path.join(data_dir, filename)
    if os.path.exists(dest_path) and os.path.getsize(dest_path) > 1024:
        print(f"{filename} already exists. Skipping.")
        continue
        
    print(f"\nDownloading {filename}...")
    cmd = ["aria2c", "-x", "16", "-s", "16", "-k", "1M", "--summary-interval=5", "--dir", data_dir, "--out", filename, url]
    subprocess.run(cmd)

print("\nDownloads completed successfully!")



04/06 07:49:02 [NOTICE] Downloading 1 item(s)

04/06 07:49:02 [NOTICE] Download complete: ./lanl_data/redteam.txt.gz

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
28e4f5|OK  |   0.9MiB/s|./lanl_data/redteam.txt.gz

Status Legend:
(OK):download completed.


04/06 07:49:02 [NOTICE] Downloading 1 item(s)
[#4b1fd9 12MiB/7.1GiB(0%) CN:16 DL:16MiB ETA:7m17s]
[#4b1fd9 43MiB/7.1GiB(0%) CN:16 DL:24MiB ETA:4m56s]
[#4b1fd9 79MiB/7.1GiB(1%) CN:16 DL:28MiB ETA:4m10s]
[#4b1fd9 123MiB/7.1GiB(1%) CN:16 DL:32MiB ETA:3m37s]
[#4b1fd9 168MiB/7.1GiB(2%) CN:16 DL:35MiB ETA:3m19s]
 *** Download Progress Summary as of Mon Apr  6 07:49:08 2026 *** 
[#4b1fd9 214MiB/7.1GiB(2%) CN:16 DL:37MiB ETA:3m9s]
FILE: ./lanl_data/auth.txt.gz
-------------------------------------------------------------------------------

[#4b1fd9 214MiB/7.1GiB(2%) CN:16 DL:37MiB ETA:3m9s]
[#4b1fd9 259MiB/7.1GiB(3%) CN:16 DL:38MiB ETA:3m1s]
[#4b1fd9 304M

## 3) ML Imports and Schema Definitions

This cell imports core ML/data libraries (`pandas`, `numpy`, `xgboost`, `scikit-learn`) and configures warnings.

It then defines:
- A `files` dictionary mapping each log type to its file path and column schema.
- `redteam_file`, which points to the ground-truth attack timeline.

No output is expected here; this is pipeline configuration.


In [3]:
import gc
import json
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score
import warnings
warnings.filterwarnings('ignore')

files = {
    "Auth":  (os.path.join(data_dir, 'auth.txt.gz'),  ['time', 'src_user', 'dest_user', 'src_comp', 'dest_comp', 'auth_type', 'logon_type', 'auth_orientation', 'success']),
    "Proc":  (os.path.join(data_dir, 'proc.txt.gz'),  ['time', 'src_user', 'src_comp', 'proc_name', 'start_end']),
    "Flow":  (os.path.join(data_dir, 'flows.txt.gz'), ['time', 'duration', 'src_comp', 'src_port', 'dest_comp', 'dest_port', 'protocol', 'pkt_cnt', 'byte_cnt']),
    "DNS":   (os.path.join(data_dir, 'dns.txt.gz'),   ['time', 'src_comp', 'dest_comp'])
}
redteam_file = os.path.join(data_dir, 'redteam.txt.gz')

## 4) Ground-Truth Label Source (Red Team)

This cell loads `redteam.txt.gz` into a dataframe and builds two lookup structures:
- `red_times`: all attack timestamps.
- `red_keys`: `(time, src_comp)` pairs used to mark anomalies in other logs.

Saved output indicates:
- `Loaded 749 known malicious events.`

This is the supervision anchor for later labeling.


In [16]:
print("Loading Red Team ground truth data...")
red_cols = ['time', 'src_user', 'src_comp', 'dest_comp']
redteam_df = pd.read_csv(redteam_file, names=red_cols, na_values=['?'])

red_times = set(redteam_df['time'])
red_keys = set(zip(redteam_df['time'], redteam_df['src_comp']))
print(f"Loaded {len(redteam_df)} known malicious events.")

Loading Red Team ground truth data...
Loaded 749 known malicious events.


## 5) Chunked Scanning and Weak Labeling

`scan_and_sample(...)` is the core data reduction and labeling function for very large logs.

What it does per chunk:
- Parses `time` as numeric integer.
- Finds rows whose `time` exists in red-team times.
- Further filters by `(time, src_comp)` to produce anomaly-labeled rows (`is_anomaly=1`).
- Randomly samples 0.1% benign rows (`is_anomaly=0`).
- Appends both to a reduced training dataset.

Why this design matters:
- Keeps memory usage manageable through chunking.
- Handles extreme class imbalance by retaining all matched anomalies while downsampling benign traffic.


In [17]:
import math

def scan_and_sample(filepath, columns, chunksize=15_000_000):
    print(f"\nScanning logs (~{os.path.getsize(filepath) / (1024**3):.1f} GB compressed)...")
    processed_chunks = []
    current_chunk = 0
    total_anomalies = 0
    
    for chunk in pd.read_csv(filepath, names=columns, chunksize=chunksize, dtype=str, na_values=['?']):
        current_chunk += 1
        chunk['time'] = pd.to_numeric(chunk['time'], errors='coerce').fillna(0).astype(np.int32)
        
        potential = chunk[chunk['time'].isin(red_times)]
        anomalies = pd.DataFrame()
        if not potential.empty:
            chunk_keys = list(zip(potential['time'], potential['src_comp']))
            mask = pd.Series(chunk_keys).isin(red_keys).values
            anomalies = potential[mask].copy()
            if not anomalies.empty:
                anomalies['is_anomaly'] = 1
                total_anomalies += len(anomalies)
        
        benign = chunk.sample(frac=0.001, random_state=42).copy()
        benign['is_anomaly'] = 0
        
        processed_chunks.append(pd.concat([anomalies, benign]))
        if current_chunk % 5 == 0:
            print(f"  -> Scanned Chunk {current_chunk} | Anomalies Found: {total_anomalies}")

    return pd.concat(processed_chunks, ignore_index=True)

## 6) XGBoost Training and Saving

`train_and_save_model(...)` trains a binary classifier and saves both model and metadata.

Key technical details:
- Stratified train/test split (`test_size=0.2`).
- Class-imbalance handling via `scale_pos_weight = sqrt(neg/pos)`.
- XGBoost configured with histogram trees and native categorical support.
- Evaluation prints classification report and AUPRC.
- Saves model JSON and metadata JSON in `./saved_models`.


In [18]:
def train_and_save_model(df, model_name, max_depth=6):
    print(f"\n{'='*40}")
    print(f" TRAINING MODEL: {model_name}")
    print(f"{'='*40}")
    
    X = df.drop(columns=['is_anomaly'])
    y = df['is_anomaly'].astype(np.int8)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    # Calculate the raw extreme imbalance
    raw_weight = len(y_train[y_train == 0]) / max(len(y_train[y_train == 1]), 1)

    # Tame the weight using a square root to balance Precision and Recall
    scale_weight = math.sqrt(raw_weight)
    
    # Enable categorical support natively in XGBoost
    xgb_params = {
        'objective': 'binary:logistic',
        'tree_method': 'hist',
        'device': 'cuda', # Change to 'cpu' if not using a GPU
        'enable_categorical': True, 
        'n_estimators': 300,
        'learning_rate': 0.05,
        'max_depth': max_depth,
        'scale_pos_weight': scale_weight,
        'random_state': 42
    }
    
    model = xgb.XGBClassifier(**xgb_params)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    print(f"\n--- {model_name} Classification Report ---")
    print(classification_report(y_test, y_pred))
    
    auprc = average_precision_score(y_test, y_pred_proba)
    print(f"Area Under Precision-Recall Curve (AUPRC): {auprc:.4f}")
    
    # Save Model
    model_path = os.path.join(model_dir, f'lanl_{model_name.lower()}_xgb.json')
    model.save_model(model_path)
    
    # Save Metadata
    metadata = {
        "model_name": f"LANL_{model_name}_Anomaly_Detector",
        "algorithm": "XGBoost",
        "features": list(X_train.columns),
        "auprc_score": round(float(auprc), 4)
    }
    with open(os.path.join(model_dir, f'{model_name.lower()}_metadata.json'), 'w') as f:
        json.dump(metadata, f, indent=4)
        
    print(f"-> Successfully saved {model_name} model & metadata.")

## 11) Network Flow Feature Engineering

`engineer_flow(...)` builds richer flow-centric features:
- Numeric conversion of duration, ports, packet count, and bytes.
- `hour_of_day` and `is_off_hours`.
- `is_lateral_movement` from host mismatch.
- `bytes_per_packet` as a traffic-shape ratio.
- Duration flags: `is_short_flow`, `is_long_flow`.
- `protocol` as categorical.

This representation captures both temporal and volumetric network behavior.


In [11]:
# ==========================================
# 3. NETWORK FLOWS
# ==========================================
def engineer_flow(df):
    for col in ['duration', 'src_port', 'dest_port', 'pkt_cnt', 'byte_cnt']:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype(np.float32)

    df['hour_of_day'] = (df['time'] % 86400) // 3600
    df['is_off_hours'] = df['hour_of_day'].apply(lambda x: 1 if x < 8 or x > 18 else 0).astype(np.int8)
    df['is_lateral_movement'] = np.where(df['dest_comp'].notna() & (df['dest_comp'] != "?"), (df['src_comp'] != df['dest_comp']).astype(int), 0).astype(np.int8)
    
    df['bytes_per_packet'] = np.where(df['pkt_cnt'] > 0, df['byte_cnt'] / df['pkt_cnt'], 0).astype(np.float32)
    df['is_short_flow'] = np.where(df['duration'] < 2.0, 1, 0).astype(np.int8)
    df['is_long_flow'] = np.where(df['duration'] > 60.0, 1, 0).astype(np.int8)
    df['protocol'] = df['protocol'].fillna("Missing").astype(str).astype('category')
        
    df.drop(columns=['time', 'src_comp', 'dest_comp'], inplace=True, errors='ignore')
    return df

## 12) Flow Training Results (Interpretation)

Saved output for Flow logs shows:
- Scan over ~1.0 GB compressed data.
- 575 matched anomalies discovered during scanning.
- Strong anomaly detection metrics on test data:
  - Class 1 precision: `0.69`
  - Class 1 recall: `0.93`
  - AUPRC: `0.7820`

Technical interpretation:
- Flow features capture red-team behavior much better than process features in this notebook.
- High recall indicates most labeled malicious flows were found; moderate precision indicates some false positives, which is common in security detection trade-offs.


In [12]:
print("\n>>> Processing Network Flow Logs...")
df_flow = scan_and_sample(*files["Flow"])
df_flow = engineer_flow(df_flow)
train_and_save_model(df_flow, "Flow", max_depth=8) # Flows need deeper trees due to complexity
del df_flow
gc.collect()


>>> Processing Network Flow Logs...

Scanning logs (~1.0 GB compressed)...
  -> Scanned Chunk 5 | Anomalies Found: 575

 TRAINING MODEL: Flow

--- Flow Classification Report ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     25996
           1       0.69      0.93      0.80       132

    accuracy                           1.00     26128
   macro avg       0.85      0.96      0.90     26128
weighted avg       1.00      1.00      1.00     26128

Area Under Precision-Recall Curve (AUPRC): 0.7820
-> Successfully saved Flow model & metadata.


83

## Summary

From the saved execution evidence:
- Data preparation and downloads completed.
- Ground truth contains 749 malicious events.
- Flow model achieved useful anomaly discrimination (AUPRC 0.7820) and appears to be the strongest detector in this notebook.